In [4]:
# 1st Cell
# Should Be Run 1st
# SVM / RF
import kagglehub
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report



# Download latest version
path = kagglehub.dataset_download("gpiosenka/musical-instruments-image-classification")

print("Path to dataset files:", path)

# Define classes
classes = ["Didgeridoo","Tambourine","Xylophone","acordian","alphorn","bagpipes","banjo","bongo drum",
"casaba","castanets","clarinet","clavichord","concertina","drums","dulcimer","flute","guiro","guitar",
"harmonica","harp","marakas","ocarina","piano","saxaphone","sitar","steel drum","trombone","trumpet",
 "tuba","violin"]

images = []
labels = []
features = list()
# List of subdirectories where the actual class folders are located
data_subdirs = ['train', 'valid', 'test']

# Iterate through each data subdirectory (e.g., 'train', 'valid', 'test')
for sub_dir in data_subdirs:
    current_data_path = os.path.join(path, sub_dir)

    if not os.path.exists(current_data_path):
        print(f"Warning: Data subdirectory '{sub_dir}' not found at '{current_data_path}'. Skipping.")
        continue

    for index, class_name in enumerate(classes):
        folder = os.path.join(current_data_path, class_name)

        # Check if the folder for the class actually exists before trying to list its contents
        if not os.path.exists(folder):
            print(f"Warning: Folder for class '{class_name}' not found at '{folder}' in '{sub_dir}'. Skipping.")
            continue

        for file in os.listdir(folder):
            image_path = os.path.join(folder, file)

            image = cv2.imread(image_path)

            if image is None:
                print(f"Warning: Could not read image at {image_path}. Skipping.")
                continue

            image = cv2.resize(image, (128, 128))
            images.append(image)
            labels.append(index)

images = np.array(images)
labels = np.array(labels)

print("Number of images loaded:", len(images))


for img in images:
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

  features.append(hog_features)

features = np.array(features) # Moved this line outside the loop
print("Features shape:", features.shape)


X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

svm = SVC(kernel='rbf')
svm.fit(X_train, y_train)





y_pred_rf = rf.predict(X_test)
y_pred_svm = svm.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

print("Random Forest Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("SVM Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))

print("Random Forest Report:\n", classification_report(y_test, y_pred_rf, target_names=classes))


Using Colab cache for faster access to the 'musical-instruments-image-classification' dataset.
Path to dataset files: /kaggle/input/musical-instruments-image-classification
Number of images loaded: 5093
Features shape: (5093, 1764)
Random Forest Accuracy: 0.47006869479882235
SVM Accuracy: 0.5662414131501472
Random Forest Confusion Matrix:
 [[ 2  0  0  4  4  1  1  0  1  0  0  1  0  1  1  4  0  0  0  2  0  1  0  0
   4  2  1  1  0  0]
 [ 2 38  2  1  0  0  0  0  1  0  0  2  1  1  1  0  1  0  1  1  0  0  1  0
   0  1  0  1  1  1]
 [ 0  5 22  1  0  0  0  0  3  0  0  5  1  1  0  1  0  0  1  0  1  1  0  0
   0  0  0  0  0  1]
 [ 1  2  0 31  0  0  0  0  3  0  0  1  1  0  1  2  0  0  0  1  0  0  0  0
   1  0  0  0  0  0]
 [ 4  1  0  0 17  1  0  0  0  1  0  0  0  0  0  1  0  0  0  0  0  0  0  0
   2  1  1  0  0  0]
 [ 0  3  1  2  5  2  0  0  2  0  0  0  1  0  0  2  0  1  0  1  0  0  0  0
   7  2  0  0  0  0]
 [ 1  5  1  0  1  1 15  0  0  0  0  2  0  0  0  1  0  0  0  0  0  0  0  1
   2  1  0  1 

In [5]:
# KNN Cell
# Run After 1st Cell
from sklearn.neighbors import KNeighborsClassifier

# Initialize KNN classifier
knn = KNeighborsClassifier(
    n_neighbors=5,        # Number of nearest neighbors
    metric='euclidean',   # Distance metric
    weights='distance'    # Closer neighbors have more influence
)

# Train the model
knn.fit(X_train, y_train)

# Predict on test set
y_pred_knn = knn.predict(X_test)

# Evaluate performance
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("KNN Confusion Matrix:\n", confusion_matrix(y_test, y_pred_knn))
print("KNN Classification Report:\n",
      classification_report(y_test, y_pred_knn, target_names=classes))


KNN Accuracy: 0.48773307163886165
KNN Confusion Matrix:
 [[15  0  0  0  3  2  0  0  0  0  1  1  0  0  3  2  0  0  0  1  0  0  0  0
   3  0  0  0  0  0]
 [ 3 37  0  1  0  0  0  1  3  2  0  0  6  1  1  0  1  0  0  0  0  0  0  0
   0  0  0  1  0  0]
 [ 1  1 20  1  2  0  0  1  1  0  1  6  3  0  2  1  0  0  1  0  0  1  1  0
   0  0  0  0  0  0]
 [ 7  1  0 28  1  1  0  0  1  0  1  0  3  0  0  0  0  0  0  0  0  0  1  0
   0  0  0  0  0  0]
 [ 8  0  0  0 15  0  0  0  0  0  0  1  0  0  0  2  0  0  0  0  2  0  0  0
   0  0  0  0  1  0]
 [12  1  0  2  1  4  0  0  0  0  0  0  2  1  0  2  0  1  0  0  0  1  0  0
   0  1  0  0  0  1]
 [ 5  2  0  1  0  0 17  0  1  0  0  1  1  0  1  0  0  2  0  0  0  0  0  1
   0  0  0  1  0  0]
 [ 1  0  0  2  0  0  0 24  0  0  1  0  0  0  0  1  0  0  0  0  0  0  1  0
   0  0  0  0  0  0]
 [ 3  1  2  1  0  0  0  1 25  0  1  1  4  0  0  1  0  1  0  1  1  0  0  1
   0  1  0  0  0  1]
 [ 2  1  0  0  1  3  1  1  0 14  0  0  3  0  0  0  0  0  0  0  1  0  0  0
   1  0  0  0 

In [6]:
# Extra Trees
# Run After 1st Cell
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Initialize Extra Trees classifier
extra_trees = ExtraTreesClassifier(
    n_estimators=300,        # More trees for better generalization
    max_depth=None,          # Let trees grow fully
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1                # Use all CPU cores (Colab-friendly)
)

# Train the model
extra_trees.fit(X_train, y_train)

# Predict on test set
y_pred_et = extra_trees.predict(X_test)

# Evaluate performance
print("Extra Trees Accuracy:", accuracy_score(y_test, y_pred_et))
print("Extra Trees Confusion Matrix:\n", confusion_matrix(y_test, y_pred_et))
print("Extra Trees Classification Report:\n",
      classification_report(y_test, y_pred_et, target_names=classes))


Extra Trees Accuracy: 0.5309126594700687
Extra Trees Confusion Matrix:
 [[10  0  0  1  3  0  1  0  1  0  0  3  0  0  3  1  0  0  1  2  0  0  0  0
   2  2  1  0  0  0]
 [ 0 45  1  0  1  0  0  1  2  0  0  0  1  0  0  1  0  1  0  1  0  0  0  0
   1  0  0  1  1  0]
 [ 0  3 25  0  0  0  0  0  5  0  0  5  2  0  0  0  0  0  0  0  1  0  1  0
   1  0  0  0  0  0]
 [ 0  0  1 32  0  0  0  0  3  1  0  4  0  0  0  1  0  0  0  1  0  0  0  0
   1  0  0  0  0  0]
 [ 1  0  0  0 20  0  0  0  0  1  0  0  1  0  0  0  0  1  0  0  0  0  0  0
   4  0  1  0  0  0]
 [ 2  1  2  1  5  1  0  0  2  0  0  1  2  0  0  2  0  0  0  1  0  0  0  0
   6  2  0  0  1  0]
 [ 3  5  0  0  0  0 16  0  2  0  0  2  0  0  0  0  0  0  1  0  0  0  0  1
   0  1  0  2  0  0]
 [ 0  0  0  0  0  0  0 27  1  0  0  0  0  1  0  0  0  0  0  1  0  0  0  0
   0  0  0  0  0  0]
 [ 0  1  1  0  2  0  0  0 36  0  0  2  0  0  0  0  0  1  0  1  0  0  0  1
   0  1  0  0  0  0]
 [ 0  4  0  0  0  0  1  0  3 15  0  0  2  0  0  1  0  0  0  1  0  0  0  0

In [10]:
# XGBoost Cell
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Initialize XGBoost classifier
xgb = XGBClassifier(
    n_estimators=400,           # Number of boosting trees
    max_depth=6,                # Controls model complexity
    learning_rate=0.05,         # Smaller LR = better generalization
    subsample=0.8,              # Row sampling
    colsample_bytree=0.3,       # Feature sampling (important for HOG)
    objective='multi:softmax',  # Multiclass classification
    num_class=len(classes),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

# Train the model
xgb.fit(X_train, y_train)

# Predict on test set
y_pred_xgb = xgb.predict(X_test)

# Evaluate performance
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("XGBoost Classification Report:\n",
      classification_report(y_test, y_pred_xgb, target_names=classes))


XGBoost Accuracy: 0.55053974484789
XGBoost Confusion Matrix:
 [[11  0  0  2  1  0  0  0  1  0  1  0  1  0  1  3  0  0  1  1  0  0  0  1
   3  1  2  0  0  1]
 [ 0 41  2  0  0  0  2  0  0  1  0  0  1  0  0  1  2  1  1  3  0  0  0  0
   0  0  1  1  0  0]
 [ 0  1 24  1  0  0  0  1  3  0  0  1  0  0  2  2  3  0  1  0  1  2  0  0
   1  0  0  0  0  0]
 [ 0  1  1 33  0  0  0  0  2  0  0  1  1  0  0  3  0  0  0  0  0  0  0  0
   0  1  0  0  1  0]
 [ 3  0  0  0 18  1  0  0  0  0  0  0  0  0  1  2  0  1  0  0  0  0  0  0
   0  2  0  0  1  0]
 [ 2  1  1  3  2  6  0  0  1  0  0  0  0  0  1  4  0  0  0  0  0  2  0  0
   5  1  0  0  0  0]
 [ 4  3  0  0  1  0 16  0  1  1  0  0  0  0  0  2  0  0  2  0  0  0  0  0
   2  0  0  0  1  0]
 [ 0  0  0  1  0  0  0 27  0  0  0  0  0  0  0  1  0  0  0  0  1  0  0  0
   0  0  0  0  0  0]
 [ 0  0  1  1  1  1  0  0 32  0  0  0  1  0  0  0  1  0  0  1  0  1  0  2
   0  1  2  0  0  1]
 [ 0  0  0  0  0  1  0  0  1 17  0  1  3  0  0  0  0  1  0  1  0  1  0  0
   0  0  